# Cost and Latency from Line One

**Phase 00** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-00/06-cost-and-latency.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You ship your first AI feature. It works. Costs look fine in testing - each request is a few cents. Three weeks later, the team is using it heavily and your API bill is $800 for the month instead of the $80 you budgeted. You investigate and discover that a prompt template you wrote starts every response with "Of course! I'd be happy to help you with that. Here's what I found:". That preamble is 20 output tokens per request. At 50,000 requests a month, that is 1 million output tokens - $4 in extra cost every month just for the greeting. Multiplied across 10 prompt templates, you have accidentally purchased $40/month of filler text.

Meanwhile, a user-facing feature that felt "fast enough" in ...

## The Concept

### Where Latency Comes From

```
Your Code                  Anthropic API
    |                           |
    |---[1. Network: ~30ms]---->|
    |                           |--[2. Queue: 0ms-2000ms]
    |                           |--[3. TTFT: 200ms-800ms]
    |<--[first token]-----------|
    |                           |--[4. Generation: 50ms per 100 tokens]
    |<--[last token]------------|
    |---[5. Your processing]--->
    |
    v
 Wall-clock latency = 1 + 2 + 3 + 4 + 5
```

**What you control:**
- Request size (fewer input tokens = less to process)
- Output length (fewer requested to...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 06: Cost and Latency from Line One
Phase 00: Setup and Mindset

A CostTracker class that wraps the Anthropic API client to record:
  - Wall-clock latency per request
  - Time-to-first-token (TTFT) for streaming requests
  - Input and output token counts from response.usage
  - Per-request cost and running totals
  - Monthly cost projections

Usage:
    export ANTHROPIC_API_KEY=sk-ant-...
    python code/main.py
"""

import os
import time
from dataclasses import dataclass, field
from typing import Any

import anthropic

### Configuration

In [ ]:
MODEL = "claude-3-5-haiku-20241022"

# Pricing in USD per million tokens.
# Source: https://www.anthropic.com/pricing
# Verify before shipping - prices change.
PRICING = {
    "input_per_million": 0.80,
    "output_per_million": 4.00,
    "cache_read_per_million": 0.08,
    "cache_write_per_million": 1.00,
}

### Data structures

In [ ]:
@dataclass
class RequestRecord:
    """Immutable record of a single API request's cost and timing data."""
    prompt_preview: str           # first 80 chars of the first user message
    input_tokens: int
    output_tokens: int
    cache_read_tokens: int
    input_cost_usd: float         # cost of fresh (non-cached) input tokens
    output_cost_usd: float
    cache_cost_usd: float
    total_cost_usd: float
    wall_clock_seconds: float     # total time from call start to last token
    ttft_seconds: float | None    # time to first token (streaming only)

    def display(self) -> str:
        ttft_str = f"  ttft={self.ttft_seconds:.2f}s" if self.ttft_seconds else ""
        return (
            f"  cost=${self.total_cost_usd:.6f}  "
            f"in={self.input_tokens} out={self.output_tokens}  "
            f"wall={self.wall_clock_seconds:.2f}s{ttft_str}\n"
            f"  prompt: \"{self.prompt_preview}...\""
        )

### CostTracker

In [ ]:
@dataclass
class CostTracker:
    """
    Wraps the Anthropic API to record cost and latency for every call.

    All calls append a RequestRecord to self.records.
    Call summary() to see aggregate stats and monthly projections.
    """
    records: list[RequestRecord] = field(default_factory=list)
    _client: anthropic.Anthropic = field(
        default_factory=lambda: anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
    )

    def _compute_cost(self, usage: Any) -> tuple[float, float, float]:
        """
        Compute (input_cost, output_cost, cache_cost) from a usage object.

        Separates fresh input tokens from cache-read tokens so each is
        priced at the correct rate.
        """
        input_tokens = getattr(usage, "input_tokens", 0) or 0
        output_tokens = getattr(usage, "output_tokens", 0) or 0
        cache_read_tokens = getattr(usage, "cache_read_input_tokens", 0) or 0

        fresh_input = max(0, input_tokens - cache_read_tokens)

        input_cost = (fresh_input / 1_000_000) * PRICING["input_per_million"]
        output_cost = (output_tokens / 1_000_000) * PRICING["output_per_million"]
        cache_cost = (cache_read_tokens / 1_000_000) * PRICING["cache_read_per_million"]

        return input_cost, output_cost, cache_cost

    def call(
        self,
        messages: list[dict],
        system: str = "",
        max_tokens: int = 1024,
        model: str = MODEL,
    ) -> tuple[anthropic.types.Message, RequestRecord]:
        """
        Non-streaming API call with cost and wall-clock latency tracking.

        Returns:
            (response, record) - use response normally, record for instrumentation.
        """
        prompt_preview = str(messages[0].get("content", ""))[:80]
        start = time.perf_counter()

        kwargs: dict[str, Any] = {
            "model": model,
            "max_tokens": max_tokens,
            "messages": messages,
        }
        if system:
            kwargs["system"] = system

        response = self._client.messages.create(**kwargs)
        elapsed = time.perf_counter() - start

        usage = response.usage
        cache_read = getattr(usage, "cache_read_input_tokens", 0) or 0
        input_cost, output_cost, cache_cost = self._compute_cost(usage)

        record = RequestRecord(
            prompt_preview=prompt_preview,
            input_tokens=usage.input_tokens,
            output_tokens=usage.output_tokens,
            cache_read_tokens=cache_read,
            input_cost_usd=input_cost,
            output_cost_usd=output_cost,
            cache_cost_usd=cache_cost,
            total_cost_usd=input_cost + output_cost + cache_cost,
            wall_clock_seconds=elapsed,
            ttft_seconds=None,
        )
        self.records.append(record)
        return response, record

    def call_streaming(
        self,
        messages: list[dict],
        system: str = "",
        max_tokens: int = 1024,
        model: str = MODEL,
        print_output: bool = True,
    ) -> tuple[str, RequestRecord]:
        """
        Streaming API call that measures time-to-first-token (TTFT).

        Returns:
            (full_text, record) - use full_text as the response content.
        """
        prompt_preview = str(messages[0].get("content", ""))[:80]
        start = time.perf_counter()
        ttft: float | None = None
        text_parts: list[str] = []

        kwargs: dict[str, Any] = {
            "model": model,
            "max_tokens": max_tokens,
            "messages": messages,
        }
        if system:
            kwargs["system"] = system

        with self._client.messages.stream(**kwargs) as stream:
            for text in stream.text_stream:
                if ttft is None:
                    ttft = time.perf_counter() - start
                text_parts.append(text)
                if print_output:
                    print(text, end="", flush=True)
            if print_output:
                print()  # newline after streaming output
            final_message = stream.get_final_message()
            usage = final_message.usage

        elapsed = time.perf_counter() - start
        full_text = "".join(text_parts)

        cache_read = getattr(usage, "cache_read_input_tokens", 0) or 0
        input_cost, output_cost, cache_cost = self._compute_cost(usage)

        record = RequestRecord(
            prompt_preview=prompt_preview,
            input_tokens=usage.input_tokens,
            output_tokens=usage.output_tokens,
            cache_read_tokens=cache_read,
            input_cost_usd=input_cost,
            output_cost_usd=output_cost,
            cache_cost_usd=cache_cost,
            total_cost_usd=input_cost + output_cost + cache_cost,
            wall_clock_seconds=elapsed,
            ttft_seconds=ttft,
        )
        self.records.append(record)
        return full_text, record

    def summary(self) -> str:
        """Return a formatted cost and latency summary with monthly projections."""
        if not self.records:
            return "No requests recorded."

        n = len(self.records)
        total_cost = sum(r.total_cost_usd for r in self.records)
        total_input = sum(r.input_tokens for r in self.records)
        total_output = sum(r.output_tokens for r in self.records)
        avg_latency = sum(r.wall_clock_seconds for r in self.records) / n

        ttft_records = [r for r in self.records if r.ttft_seconds is not None]
        avg_ttft_str = (
            f"{sum(r.ttft_seconds for r in ttft_records) / len(ttft_records):.2f}s"
            if ttft_records else "n/a (no streaming requests)"
        )

        output_input_ratio = total_output / max(total_input, 1)
        avg_cost = total_cost / n

        lines = [
            f"\n{'='*57}",
            f"  Cost + Latency Summary ({n} request{'s' if n != 1 else ''})",
            f"{'='*57}",
            f"  Total cost         : ${total_cost:.6f}",
            f"  Total input tokens : {total_input:,}",
            f"  Total output tokens: {total_output:,}",
            f"  Output/input ratio : {output_input_ratio:.2f}",
            f"  Avg wall-clock     : {avg_latency:.2f}s",
            f"  Avg TTFT (stream)  : {avg_ttft_str}",
            "",
            "  Per-request breakdown:",
        ]

        for i, r in enumerate(self.records, 1):
            lines.append(f"  [{i}] {r.display()}")

        lines.append("")
        lines.append(f"  Monthly projections (${avg_cost:.6f}/req average):")
        for volume in [1_000, 10_000, 100_000, 1_000_000]:
            projected = avg_cost * volume
            lines.append(f"    {volume:>10,} req/month = ${projected:>8.2f}/month")

        if output_input_ratio > 0.5:
            lines.append("")
            lines.append("  Note: output/input ratio > 0.5 - check for verbose preambles.")
            lines.append("  Output tokens cost 5x more than input. Cutting output saves fast.")

        return "\n".join(lines)

### Demo

In [ ]:
def demo() -> None:
    """Run three requests showing cost and latency patterns."""
    tracker = CostTracker()

    print("Request 1: Short prompt, short response")
    print("-" * 40)
    response, record = tracker.call(
        messages=[{"role": "user", "content": "What is 2 + 2?"}],
        max_tokens=50,
    )
    print(f"Response: {response.content[0].text}")
    print(record.display())

    print("\nRequest 2: Longer prompt with preamble instruction")
    print("-" * 40)
    response2, record2 = tracker.call(
        messages=[{
            "role": "user",
            "content": (
                "List three benefits of using type hints in Python. "
                "Respond with the numbered list only. No preamble or conclusion."
            ),
        }],
        max_tokens=200,
    )
    print(f"Response: {response2.content[0].text}")
    print(record2.display())

    print("\nRequest 3: Streaming with TTFT measurement")
    print("-" * 40)
    full_text, record3 = tracker.call_streaming(
        messages=[{
            "role": "user",
            "content": "Explain why output tokens cost more than input tokens in one paragraph.",
        }],
        max_tokens=150,
        print_output=True,
    )
    print(record3.display())

    print(tracker.summary())

### Demo

In [ ]:
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("Set ANTHROPIC_API_KEY environment variable before running.")
    print("Example: export ANTHROPIC_API_KEY=sk-ant-...")
    raise SystemExit(1)
demo()

---

## Self-Check

Answer these without running code first:

**1. How much are you paying per month for this preamble text alone?**

- A. $0.08/month - output tokens are cheap and the preamble is short.
- B. $1.60/month - 20 tokens x 200,000 requests x $4.00/1M = $1.60.
- C. $0.016/month - only 20 tokens, barely measurable.
- D. $8.00/month - output costs stack multiplicatively across requests.

**2. What is the primary advantage of streaming (Approach A) for this use case?**

- A. Streaming reduces the total token count because the model generates more efficiently.
- B. Streaming reduces actual API latency because tokens are generated faster.
- C. Streaming reduces perceived latency: the user sees the first words in 0.5s instead of waiting 3s for the full response.
- D. Streaming reduces cost because cache hit rates are higher for streaming calls.

**3. Which optimization has the highest potential impact on wall-clock latency?**

- A. Reduce network latency by deploying closer to the API endpoint.
- B. Reduce TTFT by batching requests together.
- C. Reduce generation time by requesting a shorter response (e.g., 200 tokens instead of 400).
- D. Reduce TTFT by using a larger, more powerful model.

**4. Which component of the cost formula explains most of the increase?**

- A. Output tokens increased - the system prompt causes longer responses.
- B. Input tokens increased - the 2,000-token system prompt is added to every request's input.
- C. Cache overhead - the system prompt broke prompt caching.
- D. Rate limit fees - more tokens per request triggers burst pricing.

**5. What is the cost reduction factor for the system prompt portion on cached requests?**

- A. 2x cheaper - caching cuts input costs in half.
- B. 5x cheaper - cache reads are $0.08 vs $0.80, a 10x reduction, but only 85% hit rate averages to 5x.
- C. 10x cheaper - cache reads at $0.08/1M vs fresh input at $0.80/1M.
- D. Free - cached tokens are not billed.

**6. Is a 1.8 output/input ratio a problem, and what should you investigate?**

- A. No problem - a high output ratio means the model is being thorough and helpful.
- B. Probably a problem - check whether responses have verbose preambles or unnecessary padding that could be removed with better prompt instructions.
- C. No problem - output tokens are cheaper than input tokens so high output ratios reduce cost.
- D. Definitely a problem - any output/input ratio above 1.0 means the model is malfunctioning.

**7. What is the estimated monthly cost for this feature?**

- A. About $9.60/month - only counting the user query tokens.
- B. About $148/month - (3,600 input tokens x $0.80 + 400 output tokens x $4.00) / 1M x 50,000 requests.
- C. About $80/month - counting only document tokens at input pricing.
- D. About $24/month - output is always the dominant cost.

**8. What is the minimum you should log on every API call going forward?**

- A. Just the response text and whether it was successful.
- B. Input tokens, output tokens, wall-clock latency, and model used - these four fields let you diagnose both cost and latency issues.
- C. Only the response time - cost can always be derived from the pricing page later.
- D. Only the cost - latency is a frontend concern, not an API concern.

_Answers are in `checks.json` in the lesson directory._